# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varshitha922/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content item for one client on one report date. I will use the March 2026 partition for verification.

In [30]:
%pip -q install duckdb

import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + token + "')"
)

march = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

print("Date check:")
print(con.sql(f"""
    SELECT COUNT(*) AS rows,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {march}
""").df())

print("Grain check (should be empty):")
print(con.sql(f"""
    SELECT report_date,
           client_hash_id,
           content_hash_id,
           COUNT(*) AS n
    FROM {march}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df())

Date check:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      rows   min_date   max_date
0  9841378 2026-03-01 2026-03-31
Grain check (should be empty):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, n]
Index: []


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:

- gsc_impressions
- gsc_clicks
- gsc_avg_position

Label:
- Future content performance outcome

Context:

- report_date
- client_hash_id
- content_hash_id

Excluded:

- Future-period fields, because they contain information from after the prediction point.
- Private or identifying fields, because they are not needed for the analysis

In [31]:
print("Section 2 field verification:")

print(con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(gsc_impressions) AS gsc_impressions_present,
        COUNT(gsc_clicks) AS gsc_clicks_present,
        COUNT(gsc_avg_position) AS gsc_avg_position_present
    FROM {march}
""").df())

Section 2 field verification:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      rows  gsc_impressions_present  gsc_clicks_present  \
0  9841378                  9841378             9841378   

   gsc_avg_position_present  
0                   3611061  


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [32]:
print("Section 3 verification:")

print(con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items
    FROM {march}
""").df())

print("Grain check (should be empty):")

print(con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {march}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df())

print("Selected fields:")

print(con.sql(f"""
    SELECT
        COUNT(gsc_impressions) AS gsc_impressions_present,
        COUNT(gsc_clicks) AS gsc_clicks_present,
        COUNT(gsc_avg_position) AS gsc_avg_position_present
    FROM {march}
""").df())

Section 3 verification:
   total_rows   min_date   max_date  clients  content_items
0     9841378 2026-03-01 2026-03-31       55         331437
Grain check (should be empty):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, n]
Index: []
Selected fields:
   gsc_impressions_present  gsc_clicks_present  gsc_avg_position_present
0                  9841378             9841378                   3611061


 ## 4.Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [33]:
print("Section 4 data limits check:")

print(con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {march}
""").df())

Section 4 data limits check:
   total_rows  clients   min_date   max_date
0     9841378       55 2026-03-01 2026-03-31




This data cannot tell us the full performance history for every client because clients have different amounts of available history. Some early rows may not have GA4 data. The 90-day query window can also overlap with the performance snapshot window, so the time windows must be aligned before using the data for prediction.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.